# Experimental kitten state

**Publication copy.** The first configuration cell was adapted to look for input files in local `data/` and save results under `outputs/`. All subsequent mathematical and comparison code is unchanged. Requires separately obtained experimental measurements and saved BI, CO and CGAN reconstructions. The final manuscript table is produced in the late cells. Historical diagnostic blocks are retained intentionally.


# Final experimental comparison: new-bandwidth KQSE without `phi_abs_floor`

This notebook performs the controlled comparison requested for the experimental kitten data.

Both KQSE branches use **exactly the same current approximate bandwidth formula**, the same
`H_MIN = 0.01`, `H_MAX = 0.5`, data folding, sign convention, CF grid, and observable transforms.
The empirical floor `phi_abs_floor = 0.03` is **removed in both branches**.

- **KQSE (no correction):** `kappa = 1`;
- **KQSE (noise corrected):** fixed `KAPPA_NOISE = 0.9` with Gaussian deconvolution.

Thus the only difference between the two reported branches is the noise-correction step.
For the corrected branch,
\[
t_{\rm eff}=r/\kappa,
\qquad
\widehat\phi_{\rm corr}
=
\widehat\phi_Z(r/\kappa)
\exp\!\left[\frac12\left(\frac{1-\kappa}{\kappa}r\right)^2\right].
\]

`KAPPA_NOISE = 0.9` is fixed a priori and is not tuned against the approximate kitten model.
BI, CO, CGAN, and GMM-3 are not rerun or retuned.


In [ ]:
# ================================================================
# 0. Imports, Drive, paths
# ================================================================

from pathlib import Path
import hashlib
import math
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from scipy.stats import gaussian_kde
from scipy.special import eval_laguerre, eval_genlaguerre, eval_hermite, gammaln

# Publication-copy path adaptation only (no estimator changes).
# Start Jupyter from the repository root, or from notebooks/.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "notebooks").exists():
    REPO_ROOT = REPO_ROOT.parent
ROOT = REPO_ROOT / "data"
OUTDIR = REPO_ROOT / "outputs" / "experimental_kitten"
OUTDIR.mkdir(parents=True, exist_ok=True)

print("Search root:", ROOT)
print("Output directory:", OUTDIR)

SEED = 0
np.random.seed(SEED)

H_MIN = 0.01
H_MAX = 0.5
R_LIST = [0, 1, 2, 3]

# Fixed experimental noise-correction parameter. Do not tune against the kitten model.
KAPPA_NOISE = 0.9
assert 0.0 < KAPPA_NOISE <= 1.0


## 1. New-bandwidth/no-floor KQSE experimental pipeline: controlled noise-correction ablation


In [ ]:
# ================================================================
# 1A. KQSE bandwidth / CF helpers
# ================================================================

def normalize_pdf_on_grid(pdf, xgrid):
    pdf = np.asarray(pdf, dtype=float)
    pdf = np.maximum(pdf, 0.0)
    Z = np.trapezoid(pdf, xgrid)
    if (not np.isfinite(Z)) or (Z <= 0):
        raise ValueError("Bad PDF normalization.")
    return pdf / Z


def pdf_kernel_optimal_bandwidth(X_grid, X_samples):
    kde = gaussian_kde(X_samples, bw_method="silverman")
    w_values = kde.evaluate(X_grid)
    return normalize_pdf_on_grid(w_values, X_grid), float(kde.factor)


def compute_phi_at_t(w_values, X_precision, t=1.0):
    integrand = w_values * np.exp(1j * X_precision * t)
    return np.trapezoid(integrand, x=X_precision)


def compute_gaussian_phi_at_t_optimal(
    mu, nu, X_precision, X_samples, w_gaussian_values, t,
    bd_min=H_MIN, bd_max=H_MAX
):
    """
    Current approximate bandwidth formula, but WITHOUT phi_abs_floor=0.03.

    A machine-level positive guard is used only to avoid division by exact zero;
    it is not a statistical/regularization floor. The resulting raw bandwidth
    is still clipped to [H_MIN, H_MAX], exactly as requested.
    """
    rho2 = mu**2 + nu**2
    if rho2 < 1e-12 or abs(t) < 1e-12:
        return 1.0 + 0.0j, 0.0

    phi_t = compute_phi_at_t(w_gaussian_values, X_precision, t)
    abs_phi = float(np.abs(phi_t))

    # Numerical guard only; no empirical phi floor.
    abs_phi_safe = max(abs_phi, np.finfo(float).tiny)

    numer = max(0.0, 1.0 - abs_phi_safe**2)
    denom = len(X_samples) * abs_phi_safe**2

    raw_bd = (
        2.0 / (abs(t) * np.sqrt(rho2))
        * np.sqrt(numer / denom)
    )
    optimal_bd = float(np.clip(raw_bd, bd_min, bd_max))

    phi_K = np.exp(
        -optimal_bd**2 * t**2 * rho2 / 4.0
    )
    phi_nk = np.mean(
        np.exp(1j * np.asarray(X_samples) * t)
    )

    return phi_nk * phi_K, optimal_bd

def symmetrize_phi_grid(phi_grid):
    phi = np.asarray(phi_grid, dtype=complex)
    return 0.5 * (phi + np.conj(phi[::-1, ::-1]))


def centered_endpoint_excluded_grid(R, N):
    delta = 2.0 * float(R) / int(N)
    return (
        np.arange(int(N)) - (int(N) - 1) / 2.0
    ) * delta

In [ ]:
# ================================================================
# 1B. Kernel transforms
# ================================================================

def photon_kernel(alpha_complex, mu, nu, r):
    radial = np.exp(-(mu**2 + nu**2) / 4.0)

    phase = np.exp(
        -((nu - 1j * mu) / np.sqrt(2.0))
         * np.conjugate(alpha_complex)
        + ((nu + 1j * mu) / np.sqrt(2.0))
         * alpha_complex
    )

    lag = eval_laguerre(
        r, (mu**2 + nu**2) / 2.0
    )
    return radial * phase * lag


def wigner_from_phi_grid(
    phi_grid, mu_grid, nu_grid, q_grid, p_grid
):
    dmu = mu_grid[1] - mu_grid[0]
    dnu = nu_grid[1] - nu_grid[0]

    W = np.zeros(
        (len(q_grid), len(p_grid)),
        dtype=complex
    )

    MU, NU = np.meshgrid(
        mu_grid, nu_grid, indexing="ij"
    )

    for iq, q in enumerate(
        tqdm(q_grid, desc="Wigner rows")
    ):
        for ip, p in enumerate(p_grid):
            phase = np.exp(
                -1j * (MU*q + NU*p)
            )
            W[iq, ip] = (
                dmu*dnu / (2*np.pi)**2
            ) * np.sum(phi_grid * phase)

    return np.real_if_close(W)


def photon_tomogram_from_phi_grid(
    phi_grid, mu_grid, nu_grid, alpha_eval_grid, r
):
    dmu = mu_grid[1] - mu_grid[0]
    dnu = nu_grid[1] - nu_grid[0]

    WR = np.zeros(
        len(alpha_eval_grid),
        dtype=complex
    )

    MU, NU = np.meshgrid(
        mu_grid, nu_grid, indexing="ij"
    )

    for ia, alpha_eval in enumerate(
        tqdm(
            alpha_eval_grid,
            desc=f"Photon tomogram r={r}"
        )
    ):
        kernel = photon_kernel(
            alpha_eval, MU, NU, r
        )
        WR[ia] = (
            dmu*dnu / (2*np.pi)
        ) * np.sum(phi_grid * kernel)

    return np.real_if_close(WR)

In [ ]:
# ================================================================
# 1C. Experimental data mapping and kitten model CF
# ================================================================

def data_wrapper(theta_list, x_list, N_theta):
    grouped_data_dict = {
        i: [] for i in range(N_theta)
    }

    theta_mod = np.mod(theta_list, np.pi)
    theta_even_odd = (
        np.floor(theta_list / np.pi).astype(int) % 2
    )
    bin_width = np.pi / N_theta
    bin_indices = np.floor(
        theta_mod / bin_width
    ).astype(int)

    for bi, x, eo in zip(
        bin_indices, x_list, theta_even_odd
    ):
        grouped_data_dict[int(bi)].append(
            float(x) * ((-1) ** (eo+1))
        )

    grouped_theta_list = [
        (i + 0.5) * bin_width
        for i in range(N_theta)
    ]

    return grouped_theta_list, grouped_data_dict


def phi_1(theta, alpha, r2, radius, t=1, p=0.6810463):
    mu = radius * np.cos(theta)
    tau = np.sqrt(1-r2)

    c0 = tau / np.sqrt(
        tau**2 + alpha**2
    )
    c1 = alpha / np.sqrt(
        tau**2 + alpha**2
    )

    beta = np.abs(radius)
    phi0 = np.exp(
        -t**2 * beta**2 / 4
    )

    # Current +i convention.
    return phi0 * (
        1
        - p*c1**2*beta**2*t**2/2
        + 1j*np.sqrt(2)*p*c0*c1*t*mu
    )


def canonical_theta_radius_sample_sign(
    mu, nu, eps=1e-12
):
    radius = float(np.hypot(mu, nu))
    if radius < eps:
        return 0.0, 0.0, 1.0

    theta_full = float(np.arctan2(nu, mu))
    if theta_full < 0:
        theta_full += 2*np.pi

    if theta_full >= np.pi:
        return (
            theta_full - np.pi,
            radius,
            -1.0
        )

    return theta_full, radius, +1.0


def experimental_phi_point_kqse(
    mu, nu,
    grouped_data_dict,
    N_theta,
    X_precision,
    noise_correction=False,
    kappa=1.0,
):
    """New-bandwidth/no-floor KCFE at one (mu,nu), optionally with Gaussian noise deconvolution.

    The approximate bandwidth formula and clipping [H_MIN,H_MAX] are identical in both branches.
    There is no empirical phi_abs_floor. Only t -> t/kappa and the inverse Gaussian-noise
    CF factor are added when noise_correction=True.
    """
    theta, radius, sample_sign = \
        canonical_theta_radius_sample_sign(mu, nu)

    if radius < 1e-12:
        return 1.0 + 0.0j, 0.0

    if noise_correction:
        if not (0.0 < float(kappa) <= 1.0):
            raise ValueError("kappa must satisfy 0 < kappa <= 1")
        t_eff = radius / float(kappa)
    else:
        t_eff = radius

    theta_bin = int(
        np.floor(
            theta / (np.pi / N_theta)
        )
    )
    theta_bin = min(
        max(theta_bin, 0),
        N_theta - 1
    )

    xb = (
        sample_sign
        * np.asarray(
            grouped_data_dict[theta_bin],
            dtype=float
        )
    )

    w_gaussian, _ = \
        pdf_kernel_optimal_bandwidth(
            X_precision, xb
        )

    mu_unit = np.cos(theta)
    nu_unit = np.sin(theta)

    phi_hat, bandwidth = compute_gaussian_phi_at_t_optimal(
        mu_unit,
        nu_unit,
        X_precision,
        xb,
        w_gaussian,
        t=t_eff
    )

    if noise_correction and float(kappa) != 1.0:
        # Gaussian Y ~ N(0,1): phi_Y(u)=exp(-u^2/2), hence inverse CF factor exp(+u^2/2).
        u_noise = ((1.0 - float(kappa)) / float(kappa)) * radius
        phi_hat = phi_hat * np.exp(0.5 * u_noise**2)

    return phi_hat, bandwidth


def build_phi_grid_experimental_kqse(
    mu_grid,
    nu_grid,
    grouped_data_dict,
    N_theta,
    X_precision,
    noise_correction=False,
    kappa=1.0,
):
    phi_grid = np.zeros(
        (len(mu_grid), len(nu_grid)),
        dtype=complex
    )
    bw_grid = np.zeros_like(
        phi_grid.real
    )

    label = (
        f"Building experimental KQSE CF (noise corr., kappa={kappa:g})"
        if noise_correction
        else "Building experimental KQSE CF (no correction)"
    )

    for i, mu in enumerate(
        tqdm(mu_grid, desc=label)
    ):
        for j, nu in enumerate(nu_grid):
            phi_grid[i,j], bw_grid[i,j] = \
                experimental_phi_point_kqse(
                    mu, nu,
                    grouped_data_dict,
                    N_theta,
                    X_precision,
                    noise_correction=noise_correction,
                    kappa=kappa,
                )

    return phi_grid, bw_grid

def build_phi_grid_model(
    mu_grid,
    nu_grid,
    alpha_model=0.3,
    r2_model=0.925,
    p_model=0.6810463
):
    phi_grid = np.zeros(
        (len(mu_grid), len(nu_grid)),
        dtype=complex
    )

    for i, mu in enumerate(mu_grid):
        for j, nu in enumerate(nu_grid):
            radius = np.hypot(mu, nu)

            if radius < 1e-12:
                phi_grid[i,j] = 1.0 + 0.0j
            else:
                theta = np.arctan2(nu, mu)
                phi_grid[i,j] = phi_1(
                    theta,
                    alpha_model,
                    r2_model,
                    radius,
                    t=1,
                    p=p_model
                )

    return phi_grid


def enforce_phi_origin(
    phi_grid, mu_grid, nu_grid
):
    out = phi_grid.copy()

    i0 = np.argmin(np.abs(mu_grid))
    j0 = np.argmin(np.abs(nu_grid))

    out[i0,j0] = 1.0 + 0.0j
    return out


### Audit of the intended estimator change

This notebook changes only the experimental KQSE regularization/noise treatment relative to the previous comparison:

1. keep the current approximate bandwidth formula;
2. remove the empirical `phi_abs_floor = 0.03` in **both** KQSE branches;
3. keep `H_MIN = 0.01`, `H_MAX = 0.5`;
4. compare `kappa=1` with fixed noise correction `kappa=0.9`;
5. leave all grids, folding/sign conventions, transforms, competitor reconstructions, and metrics unchanged.


In [ ]:
# ================================================================
# 1D. Locate data and reconstruct BOTH current KQSE variants
# ================================================================

def find_one(name):
    hits = list(ROOT.rglob(name))
    if not hits:
        hits = list(
            Path("/content/drive/MyDrive").rglob(name)
        )
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]

DATA_PATH = find_one("DataPhase28.dat")
print("Experimental data:", DATA_PATH)

theta_list, x_list = np.loadtxt(
    DATA_PATH, unpack=True
)

N_theta = 20
_, grouped_data_dict = data_wrapper(
    theta_list, x_list, N_theta
)

EXP_R_PHI = 6.5
EXP_N_PHI = 61

exp_mu_grid = centered_endpoint_excluded_grid(
    EXP_R_PHI, EXP_N_PHI
)
exp_nu_grid = centered_endpoint_excluded_grid(
    EXP_R_PHI, EXP_N_PHI
)

exp_X_precision = np.linspace(
    -3.0, 3.0, 3000
)

# ---- New bandwidth, NO phi floor, no noise correction ----
phi_exp_kqse_nocorr, bw_exp_nocorr = \
    build_phi_grid_experimental_kqse(
        exp_mu_grid,
        exp_nu_grid,
        grouped_data_dict,
        N_theta,
        exp_X_precision,
        noise_correction=False,
        kappa=1.0,
    )

phi_exp_kqse_nocorr = enforce_phi_origin(
    symmetrize_phi_grid(phi_exp_kqse_nocorr),
    exp_mu_grid,
    exp_nu_grid
)

# ---- SAME new-bandwidth/no-floor KQSE + fixed Gaussian noise correction ----
phi_exp_kqse_corr, bw_exp_corr = \
    build_phi_grid_experimental_kqse(
        exp_mu_grid,
        exp_nu_grid,
        grouped_data_dict,
        N_theta,
        exp_X_precision,
        noise_correction=True,
        kappa=KAPPA_NOISE,
    )

phi_exp_kqse_corr = enforce_phi_origin(
    symmetrize_phi_grid(phi_exp_kqse_corr),
    exp_mu_grid,
    exp_nu_grid
)

# Common approximate kitten reference on the same finite CF grid.
phi_model = build_phi_grid_model(
    exp_mu_grid,
    exp_nu_grid,
    alpha_model=0.3,
    r2_model=0.925,
    p_model=0.6810463
)

phi_model = enforce_phi_origin(
    phi_model,
    exp_mu_grid,
    exp_nu_grid
)

exp_q_grid = np.linspace(-4.0, 4.0, 81)
exp_p_grid = np.linspace(-4.0, 4.0, 81)
exp_alpha_eval = np.linspace(-5.0, 5.0, 121)

W_model = wigner_from_phi_grid(
    phi_model,
    exp_mu_grid,
    exp_nu_grid,
    exp_q_grid,
    exp_p_grid
)

W_kqse_nocorr = wigner_from_phi_grid(
    phi_exp_kqse_nocorr,
    exp_mu_grid,
    exp_nu_grid,
    exp_q_grid,
    exp_p_grid
)

W_kqse_corr = wigner_from_phi_grid(
    phi_exp_kqse_corr,
    exp_mu_grid,
    exp_nu_grid,
    exp_q_grid,
    exp_p_grid
)

w_model = {
    r: photon_tomogram_from_phi_grid(
        phi_model,
        exp_mu_grid,
        exp_nu_grid,
        exp_alpha_eval,
        r
    )
    for r in R_LIST
}

w_kqse_nocorr = {
    r: photon_tomogram_from_phi_grid(
        phi_exp_kqse_nocorr,
        exp_mu_grid,
        exp_nu_grid,
        exp_alpha_eval,
        r
    )
    for r in R_LIST
}

w_kqse_corr = {
    r: photon_tomogram_from_phi_grid(
        phi_exp_kqse_corr,
        exp_mu_grid,
        exp_nu_grid,
        exp_alpha_eval,
        r
    )
    for r in R_LIST
}

print("Both new-bandwidth/no-floor KQSE experimental variants reconstructed.")
print("Noise-correction kappa =", KAPPA_NOISE)


## 2. Load the saved BI / CO / CGAN reconstructions

The main benchmark uses the same experimental files used by the coauthors' comparison:
BI `nc=10`, saved CO, saved CGAN.

In [ ]:
# ================================================================
# 2A. File loading
# ================================================================

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def locate_preferred(name, preferred_piece=None):
    hits = list(ROOT.rglob(name))
    if not hits:
        hits = list(
            Path("/content/drive/MyDrive").rglob(name)
        )
    if not hits:
        raise FileNotFoundError(name)

    if preferred_piece is not None:
        preferred = [
            p for p in hits
            if preferred_piece in str(p)
        ]
        if preferred:
            return preferred[0]

    return hits[0]


BI10_PATH = locate_preferred(
    "rho_reconstruct_fock_nc_10.pkl",
    "Lvovsky_Data_Plots"
)
CO_PATH = locate_preferred(
    "CO_rho_reconstruct_pos.pkl"
)
CGAN_PATH = locate_preferred(
    "homodyne_qst_cgan_rho.npy"
)

rho_bi_raw = np.asarray(
    load_pickle(BI10_PATH),
    dtype=complex
)

rho_co_saved = np.asarray(
    load_pickle(CO_PATH),
    dtype=complex
)

rho_cgan_raw = np.asarray(
    np.load(CGAN_PATH),
    dtype=complex
)

print("BI:", BI10_PATH, rho_bi_raw.shape)
print("CO:", CO_PATH, rho_co_saved.shape)
print("CGAN:", CGAN_PATH, rho_cgan_raw.shape)

## 3. CO position kernel \(\to\) Fock basis and convention mapping

In [ ]:
# ================================================================
# 3A. Harmonic-oscillator basis and CO recovery
# ================================================================

def ho_basis_matrix(x_grid, dim):
    x = np.asarray(x_grid, float)

    Psi = np.empty(
        (dim, len(x)),
        dtype=float
    )

    for n in range(dim):
        norm = (
            np.pi**(-0.25)
            * np.exp(
                -0.5 * (
                    n*np.log(2.0)
                    + gammaln(n+1)
                )
            )
        )

        Psi[n] = (
            norm
            * eval_hermite(n, x)
            * np.exp(-0.5*x**2)
        )

    return Psi


def position_kernel_from_fock(
    rho_fock, x_grid
):
    rho_fock = np.asarray(
        rho_fock, complex
    )
    Psi = ho_basis_matrix(
        x_grid,
        rho_fock.shape[0]
    )
    return Psi.T @ rho_fock @ Psi


def recover_fock_from_position_kernel(
    rho_xx, x_grid, dim
):
    K = np.asarray(rho_xx, complex)
    Psi = ho_basis_matrix(x_grid, dim)
    A = Psi.T
    A_pinv = np.linalg.pinv(A)

    rho_fock = (
        A_pinv @ K @ A_pinv.T
    )

    K_back = (
        A @ rho_fock @ A.T
    )

    residual = (
        np.linalg.norm(K_back-K)
        / np.linalg.norm(K)
    )

    return rho_fock, residual


CO_X_GRID = np.linspace(
    -4.0, 4.0, 401
)

rho_co_represented, co_residual = \
    recover_fock_from_position_kernel(
        rho_co_saved,
        CO_X_GRID,
        dim=10
    )

print(
    "CO recovery residual:",
    co_residual
)

assert co_residual < 1e-10

In [ ]:
# ================================================================
# 3B. Current + kitten convention
# ================================================================

def parity_transform(rho):
    rho = np.asarray(rho, complex)
    parity = (-1.0)**np.arange(
        rho.shape[0]
    )
    return (
        parity[:,None]
        * rho
        * parity[None,:]
    )


# Source-code-derived mappings.
rho_bi = parity_transform(rho_bi_raw)
rho_co = parity_transform(rho_co_represented)
rho_cgan = rho_cgan_raw.copy()

RHO_METHODS = {
    "BI": rho_bi,
    "CO": rho_co,
    "CGAN": rho_cgan,
}


def physicality_row(name, rho):
    rho = np.asarray(rho, complex)
    rho_h = 0.5*(rho + rho.conj().T)
    eig = np.linalg.eigvalsh(rho_h)

    return {
        "Method": name,
        "dim": rho.shape[0],
        "trace": float(np.real(np.trace(rho))),
        "Hermiticity_max":
            float(np.max(np.abs(
                rho-rho.conj().T
            ))),
        "min_eigenvalue":
            float(np.min(eig)),
        "purity":
            float(np.real(
                np.trace(rho_h @ rho_h)
            )),
    }


physicality_table = pd.DataFrame([
    physicality_row(k, v)
    for k, v in RHO_METHODS.items()
])

display(physicality_table)

## 4. Validated Fock converter

The photon-number transform uses analytical displacement-operator matrix elements rather than
a truncated matrix exponential.

In [ ]:
# ================================================================
# 4A. Fock -> Wigner / photon number
# ================================================================

def displacement_element(n, m, beta):
    n = int(n)
    m = int(m)
    beta = complex(beta)

    r2 = abs(beta)**2
    gaussian = np.exp(-0.5*r2)

    if n >= m:
        factorial_ratio = np.exp(
            0.5 * (
                gammaln(m+1)
                - gammaln(n+1)
            )
        )

        return (
            gaussian
            * factorial_ratio
            * beta**(n-m)
            * eval_genlaguerre(
                m, n-m, r2
            )
        )

    factorial_ratio = np.exp(
        0.5 * (
            gammaln(n+1)
            - gammaln(m+1)
        )
    )

    return (
        gaussian
        * factorial_ratio
        * (-np.conj(beta))**(m-n)
        * eval_genlaguerre(
            n, m-n, r2
        )
    )


def displacement_block(beta, dim):
    D = np.empty(
        (dim, dim),
        dtype=complex
    )

    for n in range(dim):
        for m in range(dim):
            D[n,m] = displacement_element(
                n, m, beta
            )

    return D


def wigner_from_fock_rho(
    rho, q_grid, p_grid
):
    rho = np.asarray(rho, complex)
    dim = rho.shape[0]

    parity = (-1.0)**np.arange(dim)

    W = np.empty(
        (len(q_grid), len(p_grid)),
        dtype=float
    )

    for iq, q in enumerate(
        tqdm(q_grid, desc="Fock Wigner rows")
    ):
        for ip, p in enumerate(p_grid):
            alpha = (
                q + 1j*p
            ) / np.sqrt(2.0)

            D2 = displacement_block(
                2.0*alpha, dim
            )

            displaced_parity = (
                D2 * parity[None,:]
            )

            W[iq,ip] = np.real(
                np.trace(
                    rho @ displaced_parity
                )
            ) / np.pi

    return W


def photon_tomogram_from_fock_rho(
    rho, alpha_grid, r
):
    rho = np.asarray(rho, complex)
    dim = rho.shape[0]

    out = np.empty(
        len(alpha_grid),
        dtype=float
    )

    for ia, alpha in enumerate(alpha_grid):
        v = np.array([
            displacement_element(
                r, m, alpha
            )
            for m in range(dim)
        ])

        out[ia] = np.real(
            v @ rho @ np.conj(v)
        )

    return out

In [ ]:
# ================================================================
# 4B. Mandatory converter self-test
# ================================================================

rho_vac = np.array([[1.0+0.0j]])

q_test = np.linspace(-2, 2, 9)
p_test = np.linspace(-2, 2, 9)

W_vac_num = wigner_from_fock_rho(
    rho_vac, q_test, p_test
)

Q, P = np.meshgrid(
    q_test, p_test, indexing="ij"
)

W_vac_exact = (
    np.exp(-(Q**2+P**2))/np.pi
)

vac_W_error = np.max(
    np.abs(W_vac_num-W_vac_exact)
)

a_test = np.linspace(-2, 2, 15)

vac_ph_errors = []
for r in R_LIST:
    w_num = photon_tomogram_from_fock_rho(
        rho_vac, a_test, r
    )

    w_exact = (
        np.exp(-a_test**2)
        * np.abs(a_test)**(2*r)
        / math.factorial(r)
    )

    vac_ph_errors.append(
        np.max(np.abs(w_num-w_exact))
    )

print("Vacuum Wigner error:", vac_W_error)
print("Vacuum photon errors:", vac_ph_errors)

assert vac_W_error < 1e-12
assert max(vac_ph_errors) < 1e-12

print("Converter self-test PASSED.")

## 5. Convert BI / CO / CGAN to the same observables

In [ ]:
# ================================================================
# 5A. Competitor observables + BOTH current KQSE variants
# ================================================================

KQSE_NOCORR = "KQSE (no corr., no floor)"
KQSE_CORR = rf"KQSE (noise corr., $\kappa={KAPPA_NOISE:g}$)"

W_methods = {
    KQSE_NOCORR: np.real(W_kqse_nocorr),
    KQSE_CORR: np.real(W_kqse_corr),
}

w_methods = {
    KQSE_NOCORR: {
        r: np.real(w_kqse_nocorr[r])
        for r in R_LIST
    },
    KQSE_CORR: {
        r: np.real(w_kqse_corr[r])
        for r in R_LIST
    },
}

for method, rho in RHO_METHODS.items():
    print("\nConverting", method)

    W_methods[method] = \
        wigner_from_fock_rho(
            rho,
            exp_q_grid,
            exp_p_grid
        )

    w_methods[method] = {
        r: photon_tomogram_from_fock_rho(
            rho,
            exp_alpha_eval,
            r
        )
        for r in R_LIST
    }

print("All method observables built.")


## 6. Common metrics and controlled KQSE noise-correction table


In [ ]:
# ================================================================
# 6A. ISE and squared E_inf
# ================================================================

def ise_1d(reconstruction, target, grid):
    dx = float(grid[1]-grid[0])
    return float(dx * np.sum(np.abs(np.asarray(reconstruction)-np.asarray(target))**2))


def ise_2d(reconstruction, target, x_grid, y_grid):
    dx = float(x_grid[1]-x_grid[0])
    dy = float(y_grid[1]-y_grid[0])
    return float(dx*dy*np.sum(np.abs(np.asarray(reconstruction)-np.asarray(target))**2))


def e_inf(reconstruction, target):
    return float(np.max(np.abs(np.asarray(reconstruction)-np.asarray(target))**2))


def metric_rows(method):
    rows = [{
        "Observable": "Wigner",
        "Method": method,
        "ISE": ise_2d(W_methods[method], W_model, exp_q_grid, exp_p_grid),
        "E_inf": e_inf(W_methods[method], W_model),
    }]
    for r in R_LIST:
        rows.append({
            "Observable": f"w_{r}",
            "Method": method,
            "ISE": ise_1d(w_methods[method][r], np.real(w_model[r]), exp_alpha_eval),
            "E_inf": e_inf(w_methods[method][r], np.real(w_model[r])),
        })
    return rows

observable_order = ["Wigner", "w_0", "w_1", "w_2", "w_3"]

# Primary controlled ablation requested for the manuscript decision.
KQSE_CORRECTION_LONG = pd.DataFrame(
    metric_rows(KQSE_NOCORR) + metric_rows(KQSE_CORR)
)

# Wide table: both variants side by side, plus correction/no-correction ratios.
wide = KQSE_CORRECTION_LONG.pivot(index="Observable", columns="Method", values=["ISE", "E_inf"])
KQSE_CORRECTION_TABLE = pd.DataFrame(index=observable_order)
KQSE_CORRECTION_TABLE.index.name = "Observable"
KQSE_CORRECTION_TABLE["ISE_no_corr"] = wide["ISE"][KQSE_NOCORR]
KQSE_CORRECTION_TABLE["ISE_noise_corr"] = wide["ISE"][KQSE_CORR]
KQSE_CORRECTION_TABLE["ISE_corr/no_corr"] = (
    KQSE_CORRECTION_TABLE["ISE_noise_corr"] / KQSE_CORRECTION_TABLE["ISE_no_corr"]
)
KQSE_CORRECTION_TABLE["Einf_no_corr"] = wide["E_inf"][KQSE_NOCORR]
KQSE_CORRECTION_TABLE["Einf_noise_corr"] = wide["E_inf"][KQSE_CORR]
KQSE_CORRECTION_TABLE["Einf_corr/no_corr"] = (
    KQSE_CORRECTION_TABLE["Einf_noise_corr"] / KQSE_CORRECTION_TABLE["Einf_no_corr"]
)

print("CONTROLLED CURRENT-KQSE ABLATION")
display(KQSE_CORRECTION_TABLE)

KQSE_CORRECTION_TABLE.to_csv(
    OUTDIR / "experimental_KQSE_no_corr_vs_noise_corr.csv"
)
fmt_corr = KQSE_CORRECTION_TABLE.copy()
for col in fmt_corr.columns:
    fmt_corr[col] = fmt_corr[col].map(lambda x: f"{x:.3e}")
with open(OUTDIR / "experimental_KQSE_no_corr_vs_noise_corr.tex", "w", encoding="utf-8") as f:
    f.write(fmt_corr.to_latex(escape=False))

# Initial comparison before GMM-3 is reconstructed below.
INITIAL_METHODS = [KQSE_NOCORR, KQSE_CORR, "BI", "CO", "CGAN"]
FINAL_TABLE = pd.DataFrame(sum((metric_rows(m) for m in INITIAL_METHODS), []))
FINAL_TABLE["Observable"] = pd.Categorical(
    FINAL_TABLE["Observable"], categories=observable_order, ordered=True
)
FINAL_TABLE = FINAL_TABLE.sort_values(["Observable", "Method"]).reset_index(drop=True)

display(FINAL_TABLE)
FINAL_TABLE.to_csv(
    OUTDIR / "experimental_KQSE_both_BI_CO_CGAN_ISE_Einf.csv",
    index=False
)

print("Controlled KQSE correction table saved.")


In [ ]:
# ================================================================
# 6B. Winners by metric — descriptive only
# ================================================================

for obs in observable_order:
    sub = FINAL_TABLE[FINAL_TABLE["Observable"] == obs]
    best_ise = sub.loc[sub["ISE"].idxmin()]
    best_e = sub.loc[sub["E_inf"].idxmin()]
    print(
        f"{obs:7s} | best ISE: {best_ise['Method']} ({best_ise['ISE']:.3e}) | "
        f"best E_inf: {best_e['Method']} ({best_e['E_inf']:.3e})"
    )


## 7. Diagnostic figures with both new-bandwidth/no-floor KQSE variants


In [ ]:
# ================================================================
# 7A. Wigner diagnostic: theory | no corr | noise corr | BI | CO | CGAN
# ================================================================

panels = [
    ("Theory Model", np.real(W_model)),
    ("KQSE no corr.", W_methods[KQSE_NOCORR]),
    (rf"KQSE noise corr. ($\kappa={KAPPA_NOISE:g}$)", W_methods[KQSE_CORR]),
    ("BI", W_methods["BI"]),
    ("CO", W_methods["CO"]),
    ("CGAN", W_methods["CGAN"]),
]

vmin = min(float(np.min(W)) for _, W in panels)
vmax = max(float(np.max(W)) for _, W in panels)

fig, axes = plt.subplots(2, 3, figsize=(17.0, 10.0), constrained_layout=True)
axes = axes.ravel()

for ax, (title, W) in zip(axes, panels):
    im = ax.imshow(
        np.asarray(W).T,
        origin="lower",
        extent=[exp_q_grid[0], exp_q_grid[-1], exp_p_grid[0], exp_p_grid[-1]],
        aspect="equal",
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_title(title)
    ax.set_xlabel(r"$q$")
    ax.set_ylabel(r"$p$")

cbar = fig.colorbar(im, ax=axes.tolist(), shrink=0.92, pad=0.015)
cbar.set_label(r"$W(q,p)$")

wigner_path = OUTDIR / "fig_wigner_experimental_KQSE_no_corr_vs_noise_corr.pdf"
fig.savefig(wigner_path, bbox_inches="tight")
plt.show()
print("Saved:", wigner_path)


In [ ]:
# ================================================================
# 7B. Photon diagnostic: both current KQSE variants
# ================================================================

fig, axes = plt.subplots(2, 2, figsize=(13.5, 9.5), constrained_layout=True)
axes = axes.ravel()

for ax, r in zip(axes, R_LIST):
    ax.plot(exp_alpha_eval, np.real(w_model[r]), linewidth=2.8, label="Theory Model")
    ax.plot(exp_alpha_eval, w_methods[KQSE_NOCORR][r], linestyle="--", linewidth=2.3, label="KQSE no corr.")
    ax.plot(exp_alpha_eval, w_methods[KQSE_CORR][r], linestyle="-.", linewidth=2.3,
            label=rf"KQSE noise corr. ($\kappa={KAPPA_NOISE:g}$)")
    ax.plot(exp_alpha_eval, w_methods["BI"][r], linestyle=":", linewidth=1.8, label="BI")
    ax.plot(exp_alpha_eval, w_methods["CO"][r], linestyle=(0,(5,2)), linewidth=1.8, label="CO")
    ax.plot(exp_alpha_eval, w_methods["CGAN"][r], linestyle=(0,(3,1,1,1)), linewidth=1.8, label="CGAN")
    ax.set_xlabel(r"$\mathrm{Re}\,\alpha$")
    ax.set_ylabel(rf"$\mathsf{{w}}_{{{r}}}(\alpha)$")
    ax.grid(alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=True, bbox_to_anchor=(0.5, 1.035))

photon_path = OUTDIR / "fig_photon_experimental_KQSE_no_corr_vs_noise_corr.pdf"
fig.savefig(photon_path, bbox_inches="tight")
plt.show()
print("Saved:", photon_path)


## 8. BI cutoff sensitivity

The coauthors' plotting notebook uses `rho_reconstruct_fock_nc_10.pkl`, so `nc=10` remains
the pre-existing main BI result.

However, if additional saved BI reconstructions (`nc=5,20,50,100`) exist, we inspect them
rather than ignoring them. This block does **not** replace the main BI result by whichever
cutoff happens to look best.

For every available cutoff we report physicality, distance of the low-energy \(2\times2\)
block from the current kitten model, and population above \(n=10\). Photon-number observables
\(w_0,\ldots,w_3\) are also evaluated because they are inexpensive even for the largest cutoff.

In [ ]:
# ================================================================
# 8A. Find all saved BI cutoff reconstructions
# ================================================================

def kitten_reference_rho(
    alpha_k=0.3,
    r_bs_sq=0.925,
    p_k=0.6810463
):
    t_bs = np.sqrt(1-r_bs_sq)

    c0 = t_bs / np.sqrt(
        t_bs**2 + alpha_k**2
    )
    c1 = alpha_k / np.sqrt(
        t_bs**2 + alpha_k**2
    )

    rho = np.zeros(
        (2,2),
        dtype=complex
    )

    rho[0,0] = (
        (1-p_k) + p_k*c0**2
    )
    rho[1,1] = p_k*c1**2
    rho[0,1] = p_k*c0*c1
    rho[1,0] = p_k*c0*c1

    return rho


RHO_KITTEN_PLUS = \
    kitten_reference_rho()

BI_CUTOFFS = [5, 10, 20, 50, 100]

BI_CUTOFF_RHOS = {}
bi_rows = []

for nc in BI_CUTOFFS:
    name = (
        f"rho_reconstruct_fock_nc_{nc}.pkl"
    )

    hits = list(ROOT.rglob(name))

    if not hits:
        print("Not found:", name)
        continue

    # Prefer the bayesian folder for the sensitivity series.
    preferred = [
        p for p in hits
        if "/bayesian/" in str(p).replace("\\","/")
    ]

    path = (
        preferred[0]
        if preferred
        else hits[0]
    )

    rho_raw = np.asarray(
        load_pickle(path),
        dtype=complex
    )

    rho_plus = parity_transform(
        rho_raw
    )

    BI_CUTOFF_RHOS[nc] = rho_plus

    dim = rho_plus.shape[0]

    low2 = rho_plus[:2,:2]
    low2_error = np.linalg.norm(
        low2 - RHO_KITTEN_PLUS
    )

    tail_from_10 = (
        float(np.real(
            np.trace(
                rho_plus[10:,10:]
            )
        ))
        if dim > 10
        else 0.0
    )

    rep = physicality_row(
        f"BI nc={nc}",
        rho_plus
    )

    rep.update({
        "nc": nc,
        "path": str(path),
        "low2_Frobenius_to_model":
            float(low2_error),
        "population_n_ge_10":
            tail_from_10,
    })

    bi_rows.append(rep)


BI_CUTOFF_DIAGNOSTICS = \
    pd.DataFrame(bi_rows)

display(BI_CUTOFF_DIAGNOSTICS)

BI_CUTOFF_DIAGNOSTICS.to_csv(
    OUTDIR /
    "BI_cutoff_diagnostics.csv",
    index=False
)

In [ ]:
# ================================================================
# 8B. Photon-number sensitivity for all available BI cutoffs
# ================================================================

bi_photon_rows = []

for nc, rho in BI_CUTOFF_RHOS.items():
    print("BI photon cutoff nc =", nc)

    for r in R_LIST:
        w_bi_nc = \
            photon_tomogram_from_fock_rho(
                rho,
                exp_alpha_eval,
                r
            )

        bi_photon_rows.append({
            "nc": nc,
            "Observable": f"w_{r}",
            "ISE": ise_1d(
                w_bi_nc,
                np.real(w_model[r]),
                exp_alpha_eval
            ),
            "E_inf": e_inf(
                w_bi_nc,
                np.real(w_model[r])
            ),
        })


BI_CUTOFF_PHOTON_TABLE = \
    pd.DataFrame(bi_photon_rows)

display(BI_CUTOFF_PHOTON_TABLE)

BI_CUTOFF_PHOTON_TABLE.to_csv(
    OUTDIR /
    "BI_cutoff_photon_sensitivity.csv",
    index=False
)

## 9. Reproduce the coauthors' old position-kernel orientation

This is only a diagnostic to answer which saved method was closest under the old
position-density-kernel comparison. It does not affect the main Wigner/photon benchmark.

The comparison uses the old parity-reflected kitten convention consistent with the saved
BI/CO/CGAN plotting orientation.

In [ ]:
# ================================================================
# 9A. Old position-kernel metric
# ================================================================

RHO_KITTEN_MINUS = parity_transform(
    RHO_KITTEN_PLUS
)

rho_model_pos_old = \
    position_kernel_from_fock(
        RHO_KITTEN_MINUS,
        CO_X_GRID
    )

rho_bi_pos_old = \
    position_kernel_from_fock(
        rho_bi_raw,
        CO_X_GRID
    )

# CO was saved after the plotting transform.
rho_co_pos_old = rho_co_saved

# The coauthors' plotting code applies this transform to CGAN.
rho_cgan_pos_old = \
    position_kernel_from_fock(
        rho_cgan_raw,
        CO_X_GRID
    ).T[::-1, ::-1]


def old_position_Einf(A, B):
    return float(
        np.max(
            np.abs(
                np.asarray(A)
                - np.asarray(B)
            )**2
        )
    )


OLD_POSITION_TABLE = pd.DataFrame([
    {
        "Method": "BI",
        "E_inf_position":
            old_position_Einf(
                rho_bi_pos_old,
                rho_model_pos_old
            )
    },
    {
        "Method": "CO",
        "E_inf_position":
            old_position_Einf(
                rho_co_pos_old,
                rho_model_pos_old
            )
    },
    {
        "Method": "CGAN",
        "E_inf_position":
            old_position_Einf(
                rho_cgan_pos_old,
                rho_model_pos_old
            )
    },
])

display(
    OLD_POSITION_TABLE.sort_values(
        "E_inf_position"
    )
)

OLD_POSITION_TABLE.to_csv(
    OUTDIR /
    "old_position_kernel_comparison.csv",
    index=False
)

## 10. Summary of the pre-GMM diagnostic block

The primary new outputs are:

- `experimental_KQSE_no_corr_vs_noise_corr.csv`
- `experimental_KQSE_no_corr_vs_noise_corr.tex`
- `fig_wigner_experimental_KQSE_no_corr_vs_noise_corr.pdf`
- `fig_photon_experimental_KQSE_no_corr_vs_noise_corr.pdf`

The BI-cutoff and historical position-kernel sections remain diagnostics only. They do not alter
the current-KQSE noise-correction comparison.


# FINAL REVISED FIGURES AND TABLES — GMM-3 + BOTH CURRENT KQSE VARIANTS

This block adds the same frozen experimental GMM-3 MLE baseline and produces:

1. a controlled two-row KQSE table (no correction vs noise correction);
2. an all-method table containing both KQSE variants, GMM-3, BI, CO, and CGAN;
3. diagnostic figures containing both KQSE variants;
4. **article-candidate figures** that use the noise-corrected new-bandwidth/no-floor KQSE together with GMM-3, BI, CO, and CGAN.

No competitor reconstruction is rerun or retuned.


In [ ]:
# ================================================================
# 11A. Reconstruct the frozen experimental GMM-3 baseline
# ================================================================

from scipy.integrate import simpson
from sklearn.mixture import GaussianMixture

GMM_COMPONENTS = 3

def compute_t_fourier_transform(pdf_values, X_precision, t=1.0):
    return simpson(
        np.asarray(pdf_values) * np.exp(1j * np.asarray(X_precision) * t),
        x=np.asarray(X_precision)
    )


def reconstruct_pdf_gmm_mle(
    X_samples,
    X_precision,
    gmm_components=3,
    random_state=SEED
):
    X_samples = np.asarray(X_samples, float).ravel()
    X_grid = np.asarray(X_precision, float).ravel()

    gmm = GaussianMixture(
        n_components=gmm_components,
        covariance_type="full",
        n_init=5,
        reg_covar=1e-6,
        random_state=random_state
    ).fit(X_samples[:, None])

    pdf_hat = np.exp(
        gmm.score_samples(X_grid[:, None])
    )

    Z = simpson(pdf_hat, x=X_grid)
    if (not np.isfinite(Z)) or Z <= 0:
        raise ValueError("Bad GMM-3 MLE normalization.")

    return pdf_hat / Z


def experimental_phi_point_gmm(
    mu,
    nu,
    grouped_data_dict,
    N_theta,
    X_precision,
    gmm_components=3,
    random_state=SEED
):
    theta, radius, sample_sign = \
        canonical_theta_radius_sample_sign(mu, nu)

    if radius < 1e-12:
        return 1.0 + 0.0j

    theta_bin = int(
        np.floor(theta / (np.pi / N_theta))
    )
    theta_bin = min(
        max(theta_bin, 0),
        N_theta - 1
    )

    # Same signed-radius convention as the frozen KQSE experiment.
    xb = (
        sample_sign
        * np.asarray(
            grouped_data_dict[theta_bin],
            dtype=float
        )
    )

    pdf_hat = reconstruct_pdf_gmm_mle(
        xb,
        X_precision,
        gmm_components=gmm_components,
        random_state=random_state
    )

    return compute_t_fourier_transform(
        pdf_hat,
        X_precision,
        t=radius
    )


def build_phi_grid_experimental_gmm(
    mu_grid,
    nu_grid,
    grouped_data_dict,
    N_theta,
    X_precision,
    gmm_components=3,
    random_state=SEED
):
    phi_grid = np.zeros(
        (len(mu_grid), len(nu_grid)),
        dtype=complex
    )

    for i, mu in enumerate(
        tqdm(mu_grid, desc="Building experimental CF (GMM-3 MLE)")
    ):
        for j, nu in enumerate(nu_grid):
            phi_grid[i, j] = experimental_phi_point_gmm(
                mu,
                nu,
                grouped_data_dict,
                N_theta,
                X_precision,
                gmm_components=gmm_components,
                random_state=random_state
            )

    return phi_grid


phi_exp_gmm = build_phi_grid_experimental_gmm(
    exp_mu_grid,
    exp_nu_grid,
    grouped_data_dict,
    N_theta,
    exp_X_precision,
    gmm_components=GMM_COMPONENTS,
    random_state=SEED
)

phi_exp_gmm = enforce_phi_origin(
    symmetrize_phi_grid(phi_exp_gmm),
    exp_mu_grid,
    exp_nu_grid
)

W_gmm = wigner_from_phi_grid(
    phi_exp_gmm,
    exp_mu_grid,
    exp_nu_grid,
    exp_q_grid,
    exp_p_grid
)

w_gmm = {
    r: photon_tomogram_from_phi_grid(
        phi_exp_gmm,
        exp_mu_grid,
        exp_nu_grid,
        exp_alpha_eval,
        r
    )
    for r in R_LIST
}

W_methods["GMM-3 MLE"] = np.real(W_gmm)
w_methods["GMM-3 MLE"] = {
    r: np.real(w_gmm[r])
    for r in R_LIST
}

print("Frozen GMM-3 experimental baseline reconstructed.")

In [ ]:
# ================================================================
# 11B. Validate GMM-3 against the previously frozen article numbers
# ================================================================

gmm_check_rows = [{
    "Observable": "Wigner",
    "Method": "GMM-3 MLE",
    "ISE": ise_2d(
        W_methods["GMM-3 MLE"],
        W_model,
        exp_q_grid,
        exp_p_grid
    ),
    "E_inf": e_inf(
        W_methods["GMM-3 MLE"],
        W_model
    ),
}]

for r in R_LIST:
    gmm_check_rows.append({
        "Observable": f"w_{r}",
        "Method": "GMM-3 MLE",
        "ISE": ise_1d(
            w_methods["GMM-3 MLE"][r],
            np.real(w_model[r]),
            exp_alpha_eval
        ),
        "E_inf": e_inf(
            w_methods["GMM-3 MLE"][r],
            np.real(w_model[r])
        ),
    })

GMM_CHECK_TABLE = pd.DataFrame(gmm_check_rows)
display(GMM_CHECK_TABLE)

EXPECTED_GMM = {
    "Wigner": (9.338e-3, 1.336e-2),
    "w_0":    (2.468e-2, 1.990e-2),
    "w_1":    (8.860e-3, 6.084e-3),
    "w_2":    (5.153e-3, 6.271e-3),
    "w_3":    (6.046e-3, 6.842e-3),
}

print("\nRelative difference from frozen article values:")
for _, row in GMM_CHECK_TABLE.iterrows():
    exp_ise, exp_e = EXPECTED_GMM[row["Observable"]]
    print(
        f"{row['Observable']:7s} | "
        f"ISE rel.diff = {(row['ISE']/exp_ise - 1):+.3e} | "
        f"E_inf rel.diff = {(row['E_inf']/exp_e - 1):+.3e}"
    )

In [ ]:
# ================================================================
# 11C. Final experimental tables
# ================================================================

ALL_METHODS = [
    KQSE_NOCORR,
    KQSE_CORR,
    "GMM-3 MLE",
    "BI",
    "CO",
    "CGAN",
]

EXP_ALL_METHODS_TABLE = pd.DataFrame(
    sum((metric_rows(m) for m in ALL_METHODS), [])
)
EXP_ALL_METHODS_TABLE["Observable"] = pd.Categorical(
    EXP_ALL_METHODS_TABLE["Observable"], categories=observable_order, ordered=True
)
EXP_ALL_METHODS_TABLE = EXP_ALL_METHODS_TABLE.sort_values(
    ["Observable", "Method"]
).reset_index(drop=True)

# Candidate article comparison: corrected current KQSE vs all competitors.
ARTICLE_METHODS_CORR = [
    KQSE_CORR,
    "GMM-3 MLE",
    "BI",
    "CO",
    "CGAN",
]
EXP_ARTICLE_CORR_TABLE = pd.DataFrame(
    sum((metric_rows(m) for m in ARTICLE_METHODS_CORR), [])
)
EXP_ARTICLE_CORR_TABLE["Observable"] = pd.Categorical(
    EXP_ARTICLE_CORR_TABLE["Observable"], categories=observable_order, ordered=True
)
EXP_ARTICLE_CORR_TABLE = EXP_ARTICLE_CORR_TABLE.sort_values(
    ["Observable", "Method"]
).reset_index(drop=True)

print("CURRENT KQSE: no correction vs noise correction")
display(KQSE_CORRECTION_TABLE)

print("\nALL METHODS, BOTH KQSE VARIANTS")
display(EXP_ALL_METHODS_TABLE)

print("\nARTICLE CANDIDATE: noise-corrected KQSE vs competitors")
display(EXP_ARTICLE_CORR_TABLE)

# Save CSVs.
EXP_ALL_METHODS_TABLE.to_csv(
    OUTDIR / "experimental_ALL_methods_with_KQSE_no_corr_and_noise_corr.csv",
    index=False
)
EXP_ARTICLE_CORR_TABLE.to_csv(
    OUTDIR / "experimental_article_candidate_noise_corrected_KQSE.csv",
    index=False
)

# Save LaTeX versions.
for df, filename in [
    (EXP_ALL_METHODS_TABLE, "experimental_ALL_methods_with_KQSE_no_corr_and_noise_corr.tex"),
    (EXP_ARTICLE_CORR_TABLE, "experimental_article_candidate_noise_corrected_KQSE.tex"),
]:
    fmt = df.copy()
    fmt["ISE"] = fmt["ISE"].map(lambda x: f"{x:.3e}")
    fmt["E_inf"] = fmt["E_inf"].map(lambda x: f"{x:.3e}")
    with open(OUTDIR / filename, "w", encoding="utf-8") as f:
        f.write(fmt.to_latex(index=False, escape=False))

print("Final tables saved.")


In [ ]:
# ================================================================
# 11D. DIAGNOSTIC WIGNER FIGURE — both KQSE variants + all methods
# ================================================================

panels = [
    ("(a) Theory Model", np.real(W_model)),
    ("(b) KQSE no corr.", np.real(W_methods[KQSE_NOCORR])),
    (rf"(c) KQSE noise corr. ($\kappa={KAPPA_NOISE:g}$)", np.real(W_methods[KQSE_CORR])),
    ("(d) GMM-3 MLE", np.real(W_methods["GMM-3 MLE"])),
    ("(e) Bayesian inference", np.real(W_methods["BI"])),
    ("(f) Convex optimization", np.real(W_methods["CO"])),
    ("(g) CGAN", np.real(W_methods["CGAN"])),
]

vmin = min(float(np.min(W)) for _, W in panels)
vmax = max(float(np.max(W)) for _, W in panels)

fig, axes = plt.subplots(2, 4, figsize=(20.0, 10.6), constrained_layout=True)
axes = axes.ravel()

for ax, (title, W) in zip(axes, panels):
    im = ax.imshow(
        np.asarray(W).T,
        origin="lower",
        extent=[exp_q_grid[0], exp_q_grid[-1], exp_p_grid[0], exp_p_grid[-1]],
        aspect="equal",
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_title(title, fontsize=17, pad=8)
    ax.set_xlabel(r"$q$", fontsize=17)
    ax.set_ylabel(r"$p$", fontsize=17)
    ax.tick_params(axis="both", labelsize=12)

# Last panel intentionally unused.
axes[-1].axis("off")

cbar = fig.colorbar(im, ax=axes[:-1].tolist(), shrink=0.94, pad=0.02)
cbar.set_label(r"$W(q,p)$", fontsize=17)
cbar.ax.tick_params(labelsize=12)

wigner_all_path = OUTDIR / "fig_wigner_experimental_both_KQSE_variants_all_methods.pdf"
fig.savefig(wigner_all_path, bbox_inches="tight")
plt.show()
print("Saved:", wigner_all_path)


In [ ]:
# ================================================================
# 11E. DIAGNOSTIC PHOTON FIGURE — both KQSE variants + all methods
# ================================================================

fig, axes = plt.subplots(2, 2, figsize=(14.8, 10.8), constrained_layout=True)
axes = axes.ravel()

for ax, r in zip(axes, R_LIST):
    ax.plot(exp_alpha_eval, np.real(w_model[r]), linewidth=3.0, label="Theory Model")
    ax.plot(exp_alpha_eval, np.real(w_methods[KQSE_NOCORR][r]), linestyle="--", linewidth=2.5, label="KQSE no corr.")
    ax.plot(exp_alpha_eval, np.real(w_methods[KQSE_CORR][r]), linestyle="-.", linewidth=2.5,
            label=rf"KQSE noise corr. ($\kappa={KAPPA_NOISE:g}$)")
    ax.plot(exp_alpha_eval, np.real(w_methods["GMM-3 MLE"][r]), linestyle=":", linewidth=2.3, label="GMM-3 MLE")
    ax.plot(exp_alpha_eval, np.real(w_methods["BI"][r]), linestyle=(0,(1,1)), linewidth=2.0, label="BI")
    ax.plot(exp_alpha_eval, np.real(w_methods["CO"][r]), linestyle=(0,(5,2)), linewidth=2.0, label="CO")
    ax.plot(exp_alpha_eval, np.real(w_methods["CGAN"][r]), linestyle=(0,(3,1,1,1)), linewidth=2.0, label="CGAN")
    ax.set_title(rf"$r={r}$", fontsize=19, pad=8)
    ax.set_xlabel(r"$\mathrm{Re}\,\alpha$", fontsize=18)
    ax.set_ylabel(rf"$\mathsf{{w}}_{{{r}}}(\alpha)$", fontsize=18)
    ax.tick_params(axis="both", labelsize=13)
    ax.grid(alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=True, fontsize=12.5, bbox_to_anchor=(0.5, 1.035))

photon_all_path = OUTDIR / "fig_photon_experimental_both_KQSE_variants_all_methods.pdf"
fig.savefig(photon_all_path, bbox_inches="tight")
plt.show()
print("Saved:", photon_all_path)

# ================================================================
# ARTICLE-CANDIDATE FIGURES — noise-corrected current KQSE only
# ================================================================

article_panels = [
    ("(a) Theory Model", np.real(W_model)),
    ("(b) KQSE", np.real(W_methods[KQSE_CORR])),
    ("(c) GMM-3 MLE", np.real(W_methods["GMM-3 MLE"])),
    ("(d) Bayesian inference", np.real(W_methods["BI"])),
    ("(e) Convex optimization", np.real(W_methods["CO"])),
    ("(f) CGAN", np.real(W_methods["CGAN"])),
]

vmin = min(float(np.min(W)) for _, W in article_panels)
vmax = max(float(np.max(W)) for _, W in article_panels)
fig, axes = plt.subplots(2, 3, figsize=(17.0, 10.8), constrained_layout=True)
axes = axes.ravel()
for ax, (title, W) in zip(axes, article_panels):
    im = ax.imshow(np.asarray(W).T, origin="lower",
                   extent=[exp_q_grid[0], exp_q_grid[-1], exp_p_grid[0], exp_p_grid[-1]],
                   aspect="equal", vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=21, pad=10)
    ax.set_xlabel(r"$q$", fontsize=20)
    ax.set_ylabel(r"$p$", fontsize=20)
    ax.tick_params(axis="both", labelsize=15)
cbar = fig.colorbar(im, ax=axes.tolist(), shrink=0.94, pad=0.02)
cbar.set_label(r"$W(q,p)$", fontsize=19)
cbar.ax.tick_params(labelsize=15)
article_wigner_path = OUTDIR / "fig_wigner_experimental_article_candidate_noise_corrected_2x3.pdf"
fig.savefig(article_wigner_path, bbox_inches="tight")
plt.show()
print("Saved:", article_wigner_path)

fig, axes = plt.subplots(2, 2, figsize=(14.5, 10.5), constrained_layout=True)
axes = axes.ravel()
for ax, r in zip(axes, R_LIST):
    ax.plot(exp_alpha_eval, np.real(w_model[r]), linewidth=3.0, label="Theory Model")
    ax.plot(exp_alpha_eval, np.real(w_methods[KQSE_CORR][r]), linestyle="--", linewidth=2.7, label="KQSE")
    ax.plot(exp_alpha_eval, np.real(w_methods["GMM-3 MLE"][r]), linestyle=":", linewidth=2.8, label="GMM-3 MLE")
    ax.plot(exp_alpha_eval, np.real(w_methods["BI"][r]), linestyle="-.", linewidth=2.4, label="BI")
    ax.plot(exp_alpha_eval, np.real(w_methods["CO"][r]), linestyle=(0,(5,2)), linewidth=2.4, label="CO")
    ax.plot(exp_alpha_eval, np.real(w_methods["CGAN"][r]), linestyle=(0,(3,1,1,1)), linewidth=2.4, label="CGAN")
    ax.set_title(rf"$r={r}$", fontsize=20, pad=8)
    ax.set_xlabel(r"$\mathrm{Re}\,\alpha$", fontsize=19)
    ax.set_ylabel(rf"$\mathsf{{w}}_{{{r}}}(\alpha)$", fontsize=19)
    ax.tick_params(axis="both", labelsize=14)
    ax.grid(alpha=0.25)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=True, fontsize=14, bbox_to_anchor=(0.5, 1.035))
article_photon_path = OUTDIR / "fig_photon_experimental_article_candidate_noise_corrected.pdf"
fig.savefig(article_photon_path, bbox_inches="tight")
plt.show()
print("Saved:", article_photon_path)


## Files to inspect after running

The notebook saves everything under
`final_method_comparison_noise_correction_nofloor/`.

**Primary controlled KQSE comparison:**

- `experimental_KQSE_no_corr_vs_noise_corr.csv`
- `experimental_KQSE_no_corr_vs_noise_corr.tex`

**All methods with both KQSE variants:**

- `experimental_ALL_methods_with_KQSE_no_corr_and_noise_corr.csv`
- `experimental_ALL_methods_with_KQSE_no_corr_and_noise_corr.tex`

**Article-candidate table if we decide to use the corrected branch:**

- `experimental_article_candidate_noise_corrected_KQSE.csv`
- `experimental_article_candidate_noise_corrected_KQSE.tex`

**Diagnostic figures with both KQSE variants:**

- `fig_wigner_experimental_both_KQSE_variants_all_methods.pdf`
- `fig_photon_experimental_both_KQSE_variants_all_methods.pdf`

**Article-candidate figures using the corrected current KQSE:**

- `fig_wigner_experimental_article_candidate_noise_corrected_2x3.pdf`
- `fig_photon_experimental_article_candidate_noise_corrected.pdf`

Run the notebook top-to-bottom. After the output tables are inspected, we can update the experimental subsection and numerical tables in the manuscript. Do not select or tune `KAPPA_NOISE` by minimizing the discrepancy to the approximate kitten model.
